In [92]:
# 필요한 라이브러리 불러오기
import ee  # Google Earth Engine 파이썬 API
import requests  # 웹 요청 (썸네일 이미지 다운로드에 사용)
import pandas as pd
from PIL import Image  # 이미지 파일 열고 저장
from io import BytesIO  # 이미지 바이트 데이터를 PIL로 읽기 위한 버퍼
from datetime import datetime, timedelta  # 날짜 처리용
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import csv

import logging
from logging.handlers import RotatingFileHandler


In [ ]:
# 다운로드한 이미지들 저장 디렉토리
SAVE_PAHT = 'experimet/apt_images/'
# 위경도날짜 포함 CSV 데이터 디렉토리
DATA_PATH = '../data/interim/apt/apt_with_long_lat.csv'
# 로그 저장하는 디렉토리
LOG_PATH = '../data/log/image_download/image_download.log'
LOG_FORMAT = "[%(asctime)s] [%(levelname)s] %(message)s"
logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    handlers=[
        RotatingFileHandler(LOG_PATH, maxBytes=5*1024*1024, backupCount=5), # 5MB 단위로 로그 작성 데이터를 교환
        logging.StreamHandler() #콘솔 출력 병행??
    ]
)

#error_list = []

In [94]:
df = pd.read_csv(DATA_PATH)
df.dropna(inplace=True)

In [100]:
df = df.iloc[:500]

In [96]:



# -----------------------------------------------------------
# 1. Google Earth Engine 초기화
# -----------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='aptprice-464102')
print("Google Earth Engine 초기화 완료")
# -----------------------------------------------------------
# 2. 아파트 거래 데이터 정의
# -----------------------------------------------------------
apt_transactions = df[['위도', '경도', '계약일자']].apply(
    lambda row: {
        'lat': row['위도'],
        'lon': row['경도'],
        'date': row['계약일자']
    }, axis=1
).tolist()
print("아파트 거래 데이터 정의 완료")

def process_transaction(idx, tx):
    try:
        lat = tx['lat']
        lon = tx['lon']
        tx_date = datetime.strptime(tx['date'], "%Y-%m-%d")
        start_date = (tx_date - timedelta(days=183)).strftime('%Y-%m-%d')
        end_date = (tx_date + timedelta(days=183)).strftime('%Y-%m-%d')

        center = ee.Geometry.Point([lon, lat])
        roi = center.buffer(1500).bounds()

        collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(center)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
        )

        count = collection.size().getInfo()
        if count == 0:
            logging.warning(f"실패: lat={lat}, lon={lon}, idx={idx}, date={tx['date']}, reason=이미지 없음")
            return f"[X] 이미지 없음 (index {idx}): 날짜={tx['date']}"

        image = (
            collection
            .sort('system:time_start', False) # 최신순 정렬
            .first() # 가장 최근 이미지 선택
        )

        stats = image.reduceRegion(
            reducer=ee.Reducer.percentile([2, 98]),
            geometry=roi,
            scale=10,
            maxPixels=1e8
        ).getInfo()

        if not stats:
            logging.warning(f"실패: lat={lat}, lon={lon}, idx={idx}, date={tx['date']}, reason=통계 없음")
            return f"[X] 통계 없음 (index {idx}): 날짜={tx['date']}"

        b4_min = stats.get('B4_p2', 500)
        b4_max = stats.get('B4_p98', 3500)
        b3_min = stats.get('B3_p2', 500)
        b3_max = stats.get('B3_p98', 3500)
        b2_min = stats.get('B2_p2', 500)
        b2_max = stats.get('B2_p98', 3500)

        url = image.getThumbURL({
            'region': roi,
            'format': 'jpg',
            'bands': ['B4', 'B3', 'B2'],
            'min': [b4_min, b3_min, b2_min],
            'max': [b4_max, b3_max, b2_max],
            'scale': 10
        })

        if not url or not url.startswith("https://"):
            logging.warning(f"실패: lat={lat}, lon={lon}, idx={idx}, date={tx['date']}, reason=URL 생성 실패 (index {idx})")
            #error_list.append({"lat" : lat, 'lon' : lon, 'datetime' :tx['date'], 'reason' :  f"URL 생성 실패 (index {idx})"})
            return f"[X] URL 생성 실패 (index {idx})"

        response = requests.get(url)
        if response.status_code != 200:
            logging.warning(f"실패: lat={lat}, lon={lon}, idx={idx}, date={tx['date']}, reason=URL 생성 실패 (index {idx})")
            
            #error_list.append({"lat" : lat, 'lon' : lon, 'datetime' :tx['date'], 'reason' :  f"이미지 요청 실패 (index {idx}): 상태코드 {response.status_code}"})
            return f"[X] 이미지 요청 실패 (index {idx}): 상태코드 {response.status_code}"

        img = Image.open(BytesIO(response.content))
        img.save(f"{SAVE_PAHT}apt_image_{idx}.jpg")
        return f"[✓] 이미지 저장 완료: apt_image_{idx}.jpg"

    except Exception as e:
        logging.warning(f"실패: lat={lat}, lon={lon}, idx={idx}, date={tx['date']}, reason=예외 발생 (index {idx}): {e}")
        
        #error_list.append({"lat" : lat, 'lon' : lon, 'datetime' :tx['date'], 'reason' : f"예외 발생 (index {idx}): {e}" })
        return f"[X] 예외 발생 (index {idx}): {e}"


Google Earth Engine 초기화 완료
아파트 거래 데이터 정의 완료


In [97]:

# 병렬 처리 실행
with ThreadPoolExecutor(max_workers=400) as executor:
    futures = [executor.submit(process_transaction, idx, tx) for idx, tx in enumerate(apt_transactions)]
    for future in tqdm(as_completed(futures), total=len(futures)):
        print(future.result())

  0%|                                                  | 0/1000 [00:00<?, ?it/s][2025-06-27 12:06:21,840] [WARNING] Sleeping 1.80 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:21,840] [WARNING] Sleeping 0.44 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:21,842] [WARNING] Sleeping 0.47 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:21,846] [WARNING] Sleeping 0.73 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:21,851] [WARNING] Sleeping 0.51 seconds before retry 1 of 

[X] 이미지 요청 실패 (index 94): 상태코드 429


[2025-06-27 12:06:23,885] [WARNING] Sleeping 0.13 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:23,886] [WARNING] Sleeping 1.30 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/thumbnails?fields=name&alt=json, after 429
[2025-06-27 12:06:23,891] [WARNING] Sleeping 1.28 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:23,896] [WARNING] Sleeping 1.82 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:23,898] [WARNING] Sleeping 1.90 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:

[X] 이미지 요청 실패 (index 14): 상태코드 429
[X] 이미지 요청 실패 (index 364): 상태코드 429
[X] 이미지 요청 실패 (index 29): 상태코드 429


[2025-06-27 12:06:24,582] [WARNING] Sleeping 2.59 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:24,595] [WARNING] Sleeping 4.33 seconds before retry 3 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:24,604] [WARNING] Sleeping 3.66 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:compute?prettyPrint=false&alt=json, after 429
[2025-06-27 12:06:24,604] [WARNING] Sleeping 1.20 seconds before retry 1 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/thumbnails?fields=name&alt=json, after 429
[2025-06-27 12:06:24,613] [WARNING] Sleeping 2.34 seconds before retry 2 of 5 for request: POST https://earthengine.googleapis.com/v1/projects/aptprice-464102/value:

...

[✓] 이미지 저장 완료: apt_image_723.jpg


100%|███████████████████████████████████████▉| 999/1000 [01:18<00:02,  2.74s/it]

[✓] 이미지 저장 완료: apt_image_779.jpg


100%|███████████████████████████████████████| 1000/1000 [01:19<00:00, 12.57it/s]

[✓] 이미지 저장 완료: apt_image_383.jpg
